# Практика 2. Работа с LLM через API (GigaChat + LangChain)

Исправленная рабочая версия исходного ноутбука `llm_work.ipynb`.

**Перед запуском:**
1. Укажите свой номер варианта в `VARIANT`.
2. Положите соответствующий файл `rental_XX.csv` рядом с ноутбуком.
3. Создайте `.env` со строкой `GIGA_KEY=ваш_ключ`.
4. Не загружайте `.env` в GitVerse/GitHub и никому не отправляйте API-ключ.

Запускайте ячейки сверху вниз. Ячейки, которые обращаются к GigaChat, требуют действующего API-ключа и могут расходовать грант.

## 0. Установка библиотек

Запустите эту ячейку один раз в среде, где будет выполняться ноутбук.

In [ ]:
%pip install -q langchain-gigachat langchain-core pandas python-dotenv

## 1. Импорт библиотек и подключение к GigaChat

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

load_dotenv()
API_KEY = os.getenv("GIGA_KEY")

if not API_KEY:
    raise ValueError(
        "Не найден GIGA_KEY. Создайте файл .env рядом с ноутбуком "
        "и добавьте строку GIGA_KEY=ваш_ключ."
    )

llm = GigaChat(
    credentials=API_KEY,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

print(llm.invoke("Привет! Ответь коротко.").content)

## 2. Выбор варианта и загрузка данных

Замените `01` на свой номер из ведомости. В архиве есть варианты `01`–`33`.

Код ниже пробует UTF-8, а если файл в другой кодировке — Windows-1251.

In [ ]:
VARIANT = "20"  # ОБЯЗАТЕЛЬНО замените на свой номер, например "07"

csv_path = f"rental_{VARIANT}.csv"

try:
    df = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, sep=";", encoding="cp1251")

required_columns = {"text", "amount"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"В CSV должны быть столбцы {required_columns}. Найдены: {list(df.columns)}")

if len(df) != 15:
    raise ValueError(f"Ожидается 15 заявок, а найдено {len(df)}.")

df["amount"] = pd.to_numeric(df["amount"], errors="raise").astype(int)
print(f"Загружено заявок: {len(df)}")
display(df.head())

## 3. Задание 1 — базовый промпт

In [ ]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "Ты — эксперт по анализу текстовых заявок на аренду жилья. "
    "Определи общее количество людей, которые будут проживать."
)

user_prompt = HumanMessagePromptTemplate.from_template(
    "Заявка: {text}\n\n"
    "Извлеки общее количество проживающих. "
    "Верни только одно целое число, без пояснений."
)

chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])
chain = chat_prompt | llm | StrOutputParser()

print(chain.invoke({"text": "Семья из трех человек снимает квартиру"}))  # ожидается 3

## 4. Обработка 15 заявок и расчёт точности

In [ ]:
def run_chain(chain_to_run, data, output_column):
    results = []
    for _, row in data.iterrows():
        try:
            answer = chain_to_run.invoke({"text": row["text"]})
            results.append(str(answer).strip())
        except Exception as error:
            results.append(f"ERROR: {error}")
    data[output_column] = results
    return data

def calc_accuracy(data, output_column):
    predicted = pd.to_numeric(data[output_column], errors="coerce")
    correct_mask = predicted == data["amount"]
    correct = int(correct_mask.sum())
    total = len(data)
    accuracy = correct / total if total else 0
    print(
        f"{output_column}: верных {correct} из {total}; "
        f"ошибок {total - correct}; точность {accuracy:.1%}"
    )
    return accuracy

df = run_chain(chain, df, "result")
acc_base = calc_accuracy(df, "result")
display(df[["text", "amount", "result"]])
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")

## 5. Улучшение промпта — Few-shot

Few-shot означает, что мы показываем модели несколько примеров правильных ответов. Затем проверяем её на тех же 15 заявках.

In [ ]:
few_shot_system = SystemMessagePromptTemplate.from_template(
    """Ты — эксперт по анализу заявок на аренду жилья.
Определи ОБЩЕЕ количество проживающих: взрослых и детей вместе.

Правила:
- "семья с двумя детьми" = 4 человека (2 взрослых + 2 ребёнка);
- "семья с одним ребёнком" = 3 человека;
- "пара" = 2 человека;
- "один человек" = 1 человек;
- если прямо указано число людей, используй его;
- не путай количество комнат, спальных мест и людей.

Примеры:
Заявка: "Ищем жильё на троих" -> 3
Заявка: "Двое взрослых и четыре ребенка" -> 6
Заявка: "Одна взрослая и ребенок 5 лет" -> 2

Верни ТОЛЬКО целое число, без пояснений."""
)

few_shot_prompt = ChatPromptTemplate.from_messages([few_shot_system, user_prompt])
chain_fs = few_shot_prompt | llm | StrOutputParser()

df = run_chain(chain_fs, df, "result_fewshot")
acc_fs = calc_accuracy(df, "result_fewshot")
print(f"Изменение точности: {acc_base:.1%} -> {acc_fs:.1%}")
display(df[["text", "amount", "result", "result_fewshot"]])

### Заявки, на которых улучшенный промпт ошибся

In [ ]:
pred_fs = pd.to_numeric(df["result_fewshot"], errors="coerce")
display(df[pred_fs != df["amount"]][["text", "amount", "result_fewshot"]])

## 6. Задание 2 — извлечение дополнительных полей в JSON

Нужно извлечь:
- `count` — общее количество проживающих;
- `count_adults` — количество взрослых;
- `count_children` — количество детей;
- `start_date` — дата заезда в формате ДД.ММ;
- `nights` — число ночей;
- `price_per_day` — цена в сутки;
- `remarks` — особые пожелания.

Если данных нет, указываем `null`. Для диапазонов используем правила преподавателя: самую раннюю дату, максимальное число ночей и максимальную цену.

In [ ]:
system_msg = """Ты — эксперт по извлечению информации из заявок на аренду жилья.
Извлеки данные из текста. Не придумывай отсутствующую информацию.

Правила:
- count — общее число проживающих (взрослые + дети);
- count_adults — число взрослых;
- count_children — число детей;
- start_date — дата заезда в формате ДД.ММ; если указан диапазон дат заезда, выбери самую раннюю дату;
- nights — число ночей; если указан диапазон продолжительности, выбери максимальное число;
- price_per_day — желаемая цена за сутки числом; если указан диапазон, выбери максимальную цену;
- remarks — особые пожелания (животные, детская кроватка и т.п.), одной строкой; если их нет, null;
- если поле нельзя определить из заявки, верни null;
- не считай число комнат или спальных мест количеством проживающих.

Ответь строго JSON, без Markdown и пояснений."""

human_msg = """Примеры:

Заявка: "Ищем жильё с 4.07 на 20 дней. 2 взрослых и 2 детей, до 3000 в сутки, с кошкой"
JSON: {{"count": 4, "count_adults": 2, "count_children": 2, "start_date": "04.07", "nights": 20, "price_per_day": 3000, "remarks": "с кошкой"}}

Заявка: "Снимем жильё с 1.09 по 8.09, состав семьи не указан"
JSON: {{"count": null, "count_adults": null, "count_children": null, "start_date": "01.09", "nights": 7, "price_per_day": null, "remarks": null}}

Теперь обработай заявку:
{text}

Верни JSON с теми же ключами."""

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("human", human_msg),
])
chain_json = extract_prompt | llm | JsonOutputParser()

example_result = chain_json.invoke({"text": df.loc[0, "text"]})
print(json.dumps(example_result, ensure_ascii=False, indent=2))

### 6.1. Обработка всех заявок и сохранение JSON-результатов

In [ ]:
expected_json_keys = [
    "count", "count_adults", "count_children",
    "start_date", "nights", "price_per_day", "remarks"
]

rows = []
for idx, row in df.iterrows():
    try:
        result = chain_json.invoke({"text": row["text"]})
        if not isinstance(result, dict):
            raise ValueError("Ответ модели не является JSON-объектом")
        # Добавляем отсутствующие ключи со значением None
        result = {key: result.get(key) for key in expected_json_keys}
        result["parse_error"] = None
    except Exception as error:
        result = {key: None for key in expected_json_keys}
        result["parse_error"] = str(error)
    result["idx"] = idx
    rows.append(result)

df_json = pd.DataFrame(rows).set_index("idx")
df_json.to_csv("llm_extracted.csv", encoding="utf-8-sig")
display(df_json)

## 7. Ручная разметка

Создаём таблицу, где рядом с каждой заявкой будут ответы модели. Откройте `gold_manual.csv` в Excel или Numbers и **самостоятельно проверьте каждую строку по тексту заявки**.

Исправьте значения в столбцах `count_adults`, `count_children`, `start_date`, `nights`, `price_per_day` на правильные. Не оставляйте ответы модели автоматически, если они неверны. `remarks` тоже проверьте вручную, хотя в расчёт точности по инструкции входят поля 1–4.

In [ ]:
gold = df_json.copy()
gold.insert(0, "text", df["text"])
gold.to_csv("gold_manual.csv", encoding="utf-8-sig")
print("Создан gold_manual.csv. Теперь откройте его и вручную исправьте ответы.")

### 7.1. Расчёт точности после ручной проверки

In [ ]:
# Запускайте эту ячейку ТОЛЬКО после ручного исправления и сохранения gold_manual.csv.
gold = pd.read_csv("gold_manual.csv", index_col="idx", encoding="utf-8-sig")

fields = ["count_adults", "count_children", "start_date", "nights", "price_per_day"]

def normalize_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        value = value.strip()
        if value.lower() in ("", "nan", "none", "null"):
            return None
        return value
    return value

accuracies = {}
for field in fields:
    if field not in gold.columns or field not in df_json.columns:
        print(f"{field}: столбец отсутствует — проверьте CSV")
        continue

    model_values = df_json[field].map(normalize_value)
    manual_values = gold[field].map(normalize_value)
    matches = model_values == manual_values
    accuracy = float(matches.mean())
    accuracies[field] = accuracy
    print(f"{field}: {int(matches.sum())} из {len(matches)} = {accuracy:.1%}")

if accuracies:
    print(f"Средняя точность по полям: {sum(accuracies.values()) / len(accuracies):.1%}")

## 8. Что сохранить для сдачи

- `llm_work.ipynb` — ноутбук с выполненными ячейками и результатами.
- `rental_with_results.csv` — результаты первого задания.
- `llm_extracted.csv` — JSON-поля, извлечённые моделью.
- `gold_manual.csv` — ваша ручная проверка.
- `reflection2.md` — рефлексия на 10–12 предложений.

**Важно:** в `master` должен быть ноутбук `llm_work.ipynb` и файл `reflection2.md`. По PDF отдельная ветка `llm-improve-2` должна существовать, но вливать её в `master` не требуется. Результаты и метрики указывайте только фактически полученные после запуска.